# Day 1 | 재무데이터 수집
## Korean Stock Undervaluation Analysis — Data Collection

**목표:**
- DART Open API + FinanceDataReader로 KOSPI200 + KOSDAQ100 재무데이터 수집
- 금융주 / 적자기업 / 고PER 성장주 그룹 분리
- clean_data_final.csv (133개) 확정

**데이터 흐름:**
```
FinanceDataReader (종목 리스트)
    ↓
DART API (연간 EPS, 재무제표)
    ↓
financials_v1.csv (295개)
    ↓
그룹 분리 (금융주 / 적자 / 고PER)
    ↓
clean_data_final.csv (133개)
```


## 0. 환경 세팅

In [ ]:
# 라이브러리 설치
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime

API_KEY   = "여기에_DART_API_키_입력"
BASE_PATH = '/content/drive/MyDrive/stock-analysis'


## 1. 종목 리스트 수집 (KOSPI200 + KOSDAQ100)

In [ ]:
# KOSPI200 + KOSDAQ100 종목 리스트 수집
kospi200  = fdr.StockListing('KRX-KOSPI200')[['Code', 'Name', 'Market', 'Marcap', 'Sector']]
kosdaq100 = fdr.StockListing('KRX-KOSDAQ150')[['Code', 'Name', 'Market', 'Marcap', 'Sector']]

# KOSDAQ150 중 시총 상위 100개만 사용
kosdaq100 = kosdaq100.nlargest(100, 'Marcap')

stock_list = pd.concat([kospi200, kosdaq100], ignore_index=True)
stock_list['Code'] = stock_list['Code'].astype(str).str.zfill(6)

print(f"KOSPI200 : {len(kospi200)}개")
print(f"KOSDAQ100: {len(kosdaq100)}개")
print(f"전체     : {len(stock_list)}개")
stock_list.head()


## 2. DART corp_code 매핑

In [ ]:
# DART API에서 전체 기업 corp_code 다운로드
import zipfile, io

url  = f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={API_KEY}"
resp = requests.get(url)
z    = zipfile.ZipFile(io.BytesIO(resp.content))

# XML 파싱
import xml.etree.ElementTree as ET
tree = ET.parse(z.open('CORPCODE.xml'))
root = tree.getroot()

corp_list = []
for item in root.findall('list'):
    stock_code = item.findtext('stock_code', '').strip()
    if stock_code:  # 상장기업만
        corp_list.append({
            'corp_code': item.findtext('corp_code'),
            'corp_name': item.findtext('corp_name'),
            'Code':      stock_code
        })

df_corp = pd.DataFrame(corp_list)
df_corp['Code'] = df_corp['Code'].str.zfill(6)

# stock_list와 corp_code 매핑
stock_list = stock_list.merge(df_corp[['Code', 'corp_code']], on='Code', how='left')

print(f"corp_code 매핑 성공: {stock_list['corp_code'].notna().sum()}개")
stock_list.head()


## 3. DART API 재무데이터 수집

In [ ]:
# DART fnlttSinglAcnt API로 연간 재무데이터 수집
def get_financial_data(corp_code: str, api_key: str, year: str = '2024') -> dict:
    """
    DART 단일회사 재무제표 수집
    - fs_div: CFS(연결) 우선, 없으면 OFS(별도) fallback
    - 수집 항목: EPS, 매출, 영업이익, 순이익, 자산, 부채, 자본
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json"

    result = {}
    for fs_div in ['CFS', 'OFS']:
        params = {
            'crtfc_key': api_key,
            'corp_code': corp_code,
            'bsns_year': year,
            'reprt_code': '11011',  # 사업보고서
            'fs_div': fs_div
        }
        try:
            r    = requests.get(url, params=params, timeout=10)
            data = r.json()
            if data.get('status') == '000' and data.get('list'):
                for item in data['list']:
                    acnt = item.get('account_nm', '')
                    val  = item.get('thstrm_amount', '').replace(',', '')
                    try:
                        result[acnt] = float(val)
                    except:
                        pass
                result['fs_div'] = fs_div
                break
        except:
            pass
        time.sleep(0.2)

    return result

# 전체 수집 (약 20~30분)
financials = []
total = len(stock_list)

for idx, row in stock_list.iterrows():
    corp_code = str(row['corp_code']).zfill(8) if pd.notna(row['corp_code']) else None
    if not corp_code:
        continue

    data = get_financial_data(corp_code, API_KEY)
    data.update({
        'Code':      row['Code'],
        'Name':      row['Name'],
        'Market':    row['Market'],
        'Marcap':    row['Marcap'],
        'corp_code': corp_code,
        'Sector':    row['Sector'],
    })
    financials.append(data)

    if (idx+1) % 50 == 0:
        print(f"[{idx+1}/{total}] 수집 중...")

df_fin = pd.DataFrame(financials)
print(f"\n✅ 수집 완료: {len(df_fin)}개")


## 4. EPS 컬럼 정규화 + 저장

In [ ]:
# 기업마다 다른 EPS 계정명 통일
EPS_COLS = [
    '기본주당이익', '기본주당순이익', '기본주당순이익(손실)',
    '보통주기본주당이익(손실)', '보통주 기본주당이익',
    '1. 기본주당이익', '주당순이익'
]

def extract_eps(row):
    for col in EPS_COLS:
        if col in row and pd.notna(row[col]):
            return row[col]
    return None

df_fin['EPS'] = df_fin.apply(extract_eps, axis=1)

# 저장
import os
os.makedirs(f'{BASE_PATH}/data/processed', exist_ok=True)
df_fin.to_csv(f'{BASE_PATH}/data/processed/financials_v1.csv',
              index=False, encoding='utf-8-sig')

print(f"✅ financials_v1.csv 저장: {len(df_fin)}개")
print(f"   EPS 수집 성공: {df_fin['EPS'].notna().sum()}개")


## 5. 데이터 그룹 분리

In [ ]:
# PER 계산 (현재 주가 수집 후)
import FinanceDataReader as fdr

prices = {}
for code in df_fin['Code']:
    try:
        df_p = fdr.DataReader(code, pd.Timestamp.now() - pd.Timedelta(days=5))
        if len(df_p) > 0:
            prices[code] = df_p['Close'].iloc[-1]
    except:
        pass
    time.sleep(0.05)

df_fin['price'] = df_fin['Code'].map(prices)
df_fin['PER']   = df_fin['price'] / df_fin['EPS']

# 그룹 분리
# 1) 금융주 제거
finance_sectors = ['금융', '보험', '증권', '은행']
df_finance = df_fin[df_fin['Sector'].str.contains('|'.join(finance_sectors), na=False)]
df_non_fin = df_fin[~df_fin['Sector'].str.contains('|'.join(finance_sectors), na=False)]

# 2) 적자기업 (EPS < 0)
df_loss   = df_non_fin[df_non_fin['EPS'] < 0]
df_profit = df_non_fin[df_non_fin['EPS'] > 0]

# 3) 고PER 성장주 (IQR 기준 상한 초과)
Q3  = df_profit['PER'].quantile(0.75)
IQR = df_profit['PER'].quantile(0.75) - df_profit['PER'].quantile(0.25)
per_upper = Q3 + 1.5 * IQR

df_growth = df_profit[df_profit['PER'] > per_upper]
df_main   = df_profit[df_profit['PER'] <= per_upper]

# 저장
df_main.to_csv(f'{BASE_PATH}/data/processed/clean_data_final.csv',
               index=False, encoding='utf-8-sig')
df_growth.to_csv(f'{BASE_PATH}/data/processed/growth_stocks.csv',
                 index=False, encoding='utf-8-sig')
df_loss.to_csv(f'{BASE_PATH}/data/processed/loss_stocks.csv',
               index=False, encoding='utf-8-sig')
df_finance.to_csv(f'{BASE_PATH}/data/processed/finance_data.csv',
                  index=False, encoding='utf-8-sig')

print(f"✅ 메인 분석 대상 : {len(df_main)}개")
print(f"   고PER 성장주   : {len(df_growth)}개")
print(f"   적자기업       : {len(df_loss)}개")
print(f"   금융주         : {len(df_finance)}개")
print(f"   PER 상한 기준  : {per_upper:.1f}")
